/***************************************************************************************************
Procedure:          MDD Patient
Create Date:        2025-06-26
Author:             Naveen Deevi
Description:        This table is for Bio/nomics data table creating with the help of Felice, Elene & Erwin. This specific table is for Erwin's individual ask to pull the patient level data. This table has one row for one patient.
Customer:           Felice Sheih
Affected by table(s): Patients, Patient_Providers, providers, Provider_practices, Practice, Onboarding_metadata, patient_diagnosis
                    
Used By:            Data Science
Parameter(s):       @Patient ID created date - When the patient has onboarded to the Headlamp system. 
                    @Practice Name - Practice name of the user. 
                    @Provider name - Provider name of the user. 
Inclusion Critera: 
    1. Only Patients who have MDD as the Dx.  

Exclusion Critera: 
    1. 
    
Assumptions: 
    1. Need to be confirmed. If stopped date for a condition is not available then as per common notion 12 months are added to the started date. 
    2. 
    
****************************************************************************************************
SUMMARY OF CHANGES
Date(yyyy-mm-dd)    Author              Comments
------------------- ------------------- ------------------------------------------------------------
2025-Jun-26.        Naveen.             On Jun 25, Erwin asked for removal or omitting of Non-depressive drugs. So removing them. 
***************************************************************************************************/

In [1]:
import pandas as pd
import spacy as spc
import scispacy as scspcy
# Multiple ways to get and work with home directory
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
import requests
import math
import re
from collections import Counter

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")
#files = os.listdir(os.getcwd())
#print(files)

# Step 3: Download and load the model
try:
    nlp = spc.load("en_core_sci_sm")
    
    # Test the model
    text = "Paracetamol is commonly used to treat fever and mild to moderate pain."
    doc = nlp(text)
    
    # Print results
    print("Named Entities:")
    for ent in doc.ents:
        print(f"Entity: {ent.text}, Label: {ent.label_}")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running this in a fresh Python session")
    print("2. Try restarting your Jupyter kernel")
    print("3. Verify your Python environment has write permissions")




Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi

Contents of home directory:
Current working directory: /Users/naveendeevi/Datasets
Named Entities:
Entity: Paracetamol, Label: ENTITY
Entity: treat, Label: ENTITY
Entity: fever, Label: ENTITY
Entity: mild, Label: ENTITY
Entity: moderate pain, Label: ENTITY


/opt/anaconda3/envs/NLP/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [2]:
## Functions

nlp = spc.load("en_ner_bc5cdr_md")

def text_to_disease(drug_name):
    #print(f"Processing: {drug_name}")
    
    # Construct FDA URL
    fda_url = f'https://api.fda.gov/drug/label.json?search=openfda.brand_name:"{drug_name.lower()}"'
    
    try:
        fda_resp = requests.get(fda_url).json()
        results = fda_resp.get('results', [])

        if results and 'indications_and_usage' in results[0]:
            indication_large_body = results[0]['indications_and_usage'][0]
        else:
            return "No indications info found", None
    except Exception as e:
        print(f"Error fetching FDA data for {drug_name}: {e}")
        return "No indications info found", None

    # Clean and process text
    text_clean = ''.join(c.lower() if c.isalpha() else c for c in indication_large_body)
    result = nlp(text_clean)
    result_ents = [str(ent) for ent in result.ents if str(ent).lower() != drug_name.lower()]

    # Count and sort
    counted = Counter(result_ents)
    sorted_result = sorted(counted.items(), key=lambda x: x[1], reverse=True)

    return indication_large_body, sorted_result

# Safe wrapper to ensure consistency
def safe_text_to_disease(drug_name):
    #print(drug_name)
    try:
        output = text_to_disease(drug_name)
        if isinstance(output, tuple) and len(output) == 2:
            return output
        else:
            return "No indications info found", None
    except Exception as e:
        print(f"Exception in processing {drug_name}: {e}")
        return "No indications info found", None



In [46]:
# Functions
def get_rxcui(drug_name):
    """Get RxCUI from RxNorm for a given drug name."""
    url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name}"
    print(url)
    response = requests.get(url)
    data = response.json()
    return data.get('idGroup', {}).get('rxnormId', [None])[0]

def get_drug_classes(rxcui):
    """Get drug classes for a given RxCUI using RxClass API."""
    if not rxcui:
        return None
    #if rxcui.isna():
    #    return None
    if isinstance(rxcui, float) and math.isnan(rxcui):
        return None
    
    rxcui = int(rxcui)
    
    url = f'https://rxnav.nlm.nih.gov/REST/rxclass/class/byRxcui.json?rxcui={rxcui}'
    
    response = requests.get(url)
    data = response.json()

    classes = []
    for entry in data.get('rxclassDrugInfoList', {}).get('rxclassDrugInfo', []):
        class_name = entry.get('rxclassMinConceptItem', {}).get('className')
        class_type = entry.get('rxclassMinConceptItem', {}).get('classType')
        if class_name and class_type=="ATC1-4": #ATC standard is global standard WHO standard and a little broad
            classes.append(f"{class_name} ({class_type})")  
            break
        #elif class_name and class_type=="EPC": #EPC is the FDA standard (US Standard), usually tending more towards on which body part this works on.
        #    classes.append(f"{class_name} ({class_type})")
    return list(set(classes))  # Deduplicate

def classify_drug(drug_name):
    rxcui = get_rxcui(drug_name)
    if not rxcui:
        #return f"{drug_name}: RxCUI not found"
        return None
    classes = get_drug_classes(rxcui)
    if not classes:
        #return f"{drug_name}: No class info found"
        return None
    return "; ".join(classes)

def play_music(): 
    # List of royalty-free sample audio URLs
    music_urls = [
        "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3",
        "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-2.mp3",
        "https://www.soundhelix.com/examples/mp3/SoundHelix-Song-3.mp3"
    ]
    
    # Pick one at random
    selected_url = random.choice(music_urls)
    
    # Trigger point in your logic
    print("🎵 Trigger reached, playing random music from web...")
    
    # Play
    display(Audio(url=selected_url, autoplay=True))

def countdown(t):
    while t:
        mins, secs = divmod(t, 60)
        timer = '{:02d}:{:02d}'.format(mins, secs)
        print(timer, end="\r")
        time.sleep(1)
        t -= 1

In [47]:
df = pd.read_csv('MDD_patient_per_rows_V6.csv')

In [48]:
df.shape

(3214, 23)

In [49]:
df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mdd_first_start_dt', 'rxcui', 'medication_name',
       'dose', 'ingredient_name', 'indication_terms_fda', 'number_of_refills',
       'mood_count', 'game_count', 'ah_activity_data', 'ah_sleep_data',
       'phq_count', 'hl_excercise_counts', 'hl_sleep_counts'],
      dtype='object')

In [50]:
# Mark rows where 'depress' appears (case-insensitive) with 1, else 0
df['is_depression_related'] = df['indication_terms_fda'].str.contains('depress', case=False, na=False).astype(int)

In [51]:
#df[df['is_depression_related'] == 1]

In [52]:
#df_2 = df.loc[df['is_depression_related'] == 1,['patient_id','rxcui','medication_name','number_of_refills']]
df_2 = df[['patient_id','rxcui','medication_name','number_of_refills']]

In [53]:
df_2

,patient_id,rxcui,medication_name,number_of_refills
0,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,1086768.0,vilazodone hydrochloride,1
1,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,1099870.0,divalproex sodium 500 mg delayed release oral tablet,9
2,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,135423.0,zyprexa,20
3,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,154987.0,hydroxyzine hydrochloride,2
4,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,197891.0,lithium carbonate 300 mg extended release oral tablet,1
5,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,312079.0,olanzapine 7.5 mg oral tablet,2
6,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,NaN,vilazodone hcl 10 mg tablet,1
7,MTIzNDU2Nzg5MGFiY2RlZg2ClYVvVZa3tsAwUhpKqM4UFJEIXi0EIOxLvxiKHgnvlIzzrkqPghnqgK6GKxdtoPAV117+VUZlp6pxSYkAiNC0vXfugO0dc56m7D5KsfDO,1000048.0,doxepin 10 mg oral capsule,14
8,MTIzNDU2Nzg5MGFiY2RlZg2ClYVvVZa3tsAwUhpKqM4UFJEIXi0EIOxLvxiKHgnvlIzzrkqPghnqgK6GKxdtoPAV117+VUZlp6pxSYkAiNC0vXfugO0dc56m7D5KsfDO,155137.0,sertraline hydrochloride,3
9,MTIzNDU2Nzg5MGFiY2RlZg2ClYVvVZa3tsAwUhpKqM4UFJEIXi0EIOxLvxiKHgnvlIzzrkqPghnqgK6GKxdtoPAV117+VUZlp6pxSYkAiNC0vXfugO0dc56m7D5KsfDO,197321.0,alprazolam 1 mg oral tablet,23


In [54]:
# Get unique, non-null values from df_2['rxcui']
# Unique, non-null values as sorted integers
unique_rxcui = sorted(set(df_2['rxcui'].dropna().astype(int)))

In [55]:
len(unique_rxcui)

330

In [56]:
# Get class from RXCUI. Create a dictionary mapping the RXCUI to the class

# Create dictionary 
rxcui_class_dict = {}

# Extract distinct rxcui's & Not nulls
keys = unique_rxcui
j = 0

# Print class from the rxcui and insert into the dictionary
#try:
for i in range(0,len(keys)):
    #print(i)
    print('%d of %d'%(j,len(keys)))
    rxcui_class_dict[int(keys[i])] = get_drug_classes(int(keys[i]))
    if (j>0 and j%100 == 0):
        print("Execution paused for a 3 minutes")
        #time.sleep(180)
        #countdown(60)
        # Code will continue execution after the 5-minute pause
        print("Resumed execution after 3 minutes.")
    j+=1
        # As this code is timing out, a pause of 5 mins is given to the code at 
        #for ratio in (0.00,1.00,0.10):
        #    print(ration)        
#except:        
    #play_music()
#    print('error')

0 of 330
1 of 330
2 of 330
3 of 330
4 of 330
5 of 330
6 of 330
7 of 330
8 of 330
9 of 330
10 of 330
11 of 330
12 of 330
13 of 330
14 of 330
15 of 330
16 of 330
17 of 330
18 of 330
19 of 330
20 of 330
21 of 330
22 of 330
23 of 330
24 of 330
25 of 330
26 of 330
27 of 330
28 of 330
29 of 330
30 of 330
31 of 330
32 of 330
33 of 330
34 of 330
35 of 330
36 of 330
37 of 330
38 of 330
39 of 330
40 of 330
41 of 330
42 of 330
43 of 330
44 of 330
45 of 330
46 of 330
47 of 330
48 of 330
49 of 330
50 of 330
51 of 330
52 of 330
53 of 330
54 of 330
55 of 330
56 of 330
57 of 330
58 of 330
59 of 330
60 of 330
61 of 330
62 of 330
63 of 330
64 of 330
65 of 330
66 of 330
67 of 330
68 of 330
69 of 330
70 of 330
71 of 330
72 of 330
73 of 330
74 of 330
75 of 330
76 of 330
77 of 330
78 of 330
79 of 330
80 of 330
81 of 330
82 of 330
83 of 330
84 of 330
85 of 330
86 of 330
87 of 330
88 of 330
89 of 330
90 of 330
91 of 330
92 of 330
93 of 330
94 of 330
95 of 330
96 of 330
97 of 330
98 of 330
99 of 330
100 of 330

In [57]:
unique_values = set(val for sublist in rxcui_class_dict.values() for val in sublist)

print(unique_values)

{'Gabapentinoids (ATC1-4)', 'Orexin receptor antagonists (ATC1-4)', 'Other antiepileptics (ATC1-4)', 'Antidotes (ATC1-4)', 'Imidazoline receptor agonists (ATC1-4)', 'Selective serotonin reuptake inhibitors (ATC1-4)', 'Beta blocking agents, non-selective (ATC1-4)', 'Monoamine oxidase B inhibitors (ATC1-4)', 'Centrally acting sympathomimetics (ATC1-4)', 'Centrally acting antiobesity products (ATC1-4)', 'Peripheral opioid receptor antagonists (ATC1-4)', 'Adamantane derivatives (ATC1-4)', 'Azaspirodecanedione derivatives (ATC1-4)', 'Benzodiazepine related drugs (ATC1-4)', 'Other antidepressants (ATC1-4)', 'Other antipsychotics (ATC1-4)', 'Non-selective monoamine reuptake inhibitors (ATC1-4)', 'Alpha-adrenoreceptor antagonists (ATC1-4)', 'Indole derivatives (ATC1-4)', 'Butyrophenone derivatives (ATC1-4)', 'Other general anesthetics (ATC1-4)', 'Drugs used in opioid dependence (ATC1-4)', 'Melatonin receptor agonists (ATC1-4)', 'Phenothiazines with piperazine structure (ATC1-4)', 'Phenothiazin

In [58]:
#rxcui_class_dict

In [59]:
# Map the dictionary and unpack the list values as comma-separated strings
df_2['class'] = df_2['rxcui'].map(lambda x: ', '.join(rxcui_class_dict.get(x, [])))


/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_43390/1484984407.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['class'] = df_2['rxcui'].map(lambda x: ', '.join(rxcui_class_dict.get(x, [])))


In [60]:
df_2

,patient_id,rxcui,medication_name,number_of_refills,class
0,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,1086768.0,vilazodone hydrochloride,1,Other antidepressants (ATC1-4)
1,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,1099870.0,divalproex sodium 500 mg delayed release oral tablet,9,Fatty acid derivatives (ATC1-4)
2,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,135423.0,zyprexa,20,"Diazepines, oxazepines, thiazepines and oxepines (ATC1-4)"
3,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,154987.0,hydroxyzine hydrochloride,2,Diphenylmethane derivatives (ATC1-4)
4,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,197891.0,lithium carbonate 300 mg extended release oral tablet,1,Lithium (ATC1-4)
5,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,312079.0,olanzapine 7.5 mg oral tablet,2,"Diazepines, oxazepines, thiazepines and oxepines (ATC1-4)"
6,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8OnTAPwRbhZLIjDmrqh5Km94tFnt3qQGZzrrBv+NoaAVdC84m0FgEiIRm88ZYGPtckhUotdDJlPrdzVrHn0IJ,NaN,vilazodone hcl 10 mg tablet,1,
7,MTIzNDU2Nzg5MGFiY2RlZg2ClYVvVZa3tsAwUhpKqM4UFJEIXi0EIOxLvxiKHgnvlIzzrkqPghnqgK6GKxdtoPAV117+VUZlp6pxSYkAiNC0vXfugO0dc56m7D5KsfDO,1000048.0,doxepin 10 mg oral capsule,14,Non-selective monoamine reuptake inhibitors (ATC1-4)
8,MTIzNDU2Nzg5MGFiY2RlZg2ClYVvVZa3tsAwUhpKqM4UFJEIXi0EIOxLvxiKHgnvlIzzrkqPghnqgK6GKxdtoPAV117+VUZlp6pxSYkAiNC0vXfugO0dc56m7D5KsfDO,155137.0,sertraline hydrochloride,3,Selective serotonin reuptake inhibitors (ATC1-4)
9,MTIzNDU2Nzg5MGFiY2RlZg2ClYVvVZa3tsAwUhpKqM4UFJEIXi0EIOxLvxiKHgnvlIzzrkqPghnqgK6GKxdtoPAV117+VUZlp6pxSYkAiNC0vXfugO0dc56m7D5KsfDO,197321.0,alprazolam 1 mg oral tablet,23,Benzodiazepine derivatives (ATC1-4)


In [61]:
# Count unique patient_id per class
patient_counts = df_2.groupby('class')['patient_id'].nunique().reset_index(name='patient_count')

summary = df_2.groupby('class').agg(
    patient_count=('patient_id', 'nunique'),
    total_refills=('number_of_refills', 'sum')
).reset_index()

In [62]:
pd.set_option('display.max_rows', None)  
pd.set_option('display.max_colwidth', None)   
#patient_counts.sort_values(by='patient_count',ascending=False)
summary.sort_values(by='patient_count',ascending=False)

,class,patient_count,total_refills
28,Other antidepressants (ATC1-4),269,1797
35,Selective serotonin reuptake inhibitors (ATC1-4),222,1132
0,,174,438
6,Benzodiazepine derivatives (ATC1-4),149,990
30,Other antipsychotics (ATC1-4),132,873
13,Centrally acting sympathomimetics (ATC1-4),126,665
29,Other antiepileptics (ATC1-4),117,784
15,Diphenylmethane derivatives (ATC1-4),111,296
19,Gabapentinoids (ATC1-4),94,450
5,Azaspirodecanedione derivatives (ATC1-4),77,316


In [70]:
df[df['indication_terms_fda'].isna()].shape

(526, 23)

In [71]:
df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mdd_first_start_dt', 'rxcui', 'medication_name',
       'dose', 'ingredient_name', 'indication_terms_fda', 'number_of_refills',
       'mood_count', 'game_count', 'ah_activity_data', 'ah_sleep_data',
       'phq_count', 'hl_excercise_counts', 'hl_sleep_counts'],
      dtype='object')

In [72]:
# Extracting the first word from the name of the medication name, as it seems to be most nearest to the pull indications
med_text = set(df.loc[df['indication_terms_fda'].isna(),'medication_name'])
med_text_2 = [] 
for i in med_text:
    first_word = i.strip().split()[0]
    med_text_2.append(first_word)
med_text_2 = set(med_text_2)

In [73]:
med_text_dict = {}
for i in med_text_2: 
    med_text_dict[i] = safe_text_to_disease(i)[1]

In [74]:
#med_text_dict

In [75]:
df.head(3)

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,mdd_first_start_dt,...,ingredient_name,indication_terms_fda,number_of_refills,mood_count,game_count,ah_activity_data,ah_sleep_data,phq_count,hl_excercise_counts,hl_sleep_counts
0,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8On...,Drew Walsh,PATIENT_GENDER_MALE,1994-03-16,31,"January 3, 2025, 1:29 PM",Matthew A Berger MD PC,Matthew Berger,ONBOARDED,"December 6, 2023",...,vilazodone hydrochloride,"[('depressive disorder', 2), ('mdd', 2)]",1,23,20,NaN,NaN,NaN,23.0,23.0
1,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8On...,Drew Walsh,PATIENT_GENDER_MALE,1994-03-16,31,"January 3, 2025, 1:29 PM",Matthew A Berger MD PC,Matthew Berger,ONBOARDED,"December 6, 2023",...,divalproex sodium,"[('divalproex sodium extended-release', 5), ('...",9,23,20,NaN,NaN,NaN,23.0,23.0
2,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8On...,Drew Walsh,PATIENT_GENDER_MALE,1994-03-16,31,"January 3, 2025, 1:29 PM",Matthew A Berger MD PC,Matthew Berger,ONBOARDED,"December 6, 2023",...,olanzapine pamoate,"[('schizophrenia', 4), ('schizophrenia zyprexa...",20,23,20,NaN,NaN,NaN,23.0,23.0


In [76]:
def fill_indication_from_medication(row):
    value = row['indication_terms_fda']

    # Handle if value is None or an empty list ([])
    if value is None or (isinstance(value, list) and len(value) == 0):
        for key in med_text_dict:
            if key.lower() in str(row['medication_name']).lower():
                return med_text_dict[key]
        return value  # Return original (None or empty list) if no match found
    else:
        return value  # Leave non-null values unchanged

# Apply to the DataFrame
df['indication_terms_fda'] = df.apply(fill_indication_from_medication, axis=1)


In [77]:
pattern = r"\bdepress\w*\b"

df['is_depression_related'] = df['indication_terms_fda'].astype(str).str.contains(pattern, flags=re.IGNORECASE, na=False, regex=True)

df[df['indication_terms_fda'].isna()].shape
# The indication is null now only for 137 rows, which is 4% of 3333. Hence acceptable. 

(125, 24)

In [78]:
#df.to_csv('test_mdd_patient.csv')

In [79]:
df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mdd_first_start_dt', 'rxcui', 'medication_name',
       'dose', 'ingredient_name', 'indication_terms_fda', 'number_of_refills',
       'mood_count', 'game_count', 'ah_activity_data', 'ah_sleep_data',
       'phq_count', 'hl_excercise_counts', 'hl_sleep_counts',
       'is_depression_related'],
      dtype='object')

In [80]:
df_2 = df[['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status','mood_count', 'game_count', 'ah_activity_data', 'ah_sleep_data',
       'phq_count', 'hl_excercise_counts', 'hl_sleep_counts', 'medication_name',
        'dose','is_depression_related','number_of_refills']]

In [81]:
#df_2['Medication_Name_&_Dosage'] = df_2['medication_name']+ ';;; Dose- ' +df_2['dose']
df_2['Medication_Name_&_Dosage'] = df_2['medication_name'].str.cat(df_2['dose'], sep=';;; Dose -', na_rep='*')
df_2['Medication_Name_Dosage_&_Type'] = df_2['Medication_Name_&_Dosage'].str.cat(
    df_2['is_depression_related'].astype(str), sep=';;; Depressive- ', na_rep='*')

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_8396/2372296587.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['Medication_Name_&_Dosage'] = df_2['medication_name'].str.cat(df_2['dose'], sep=';;; Dose -', na_rep='*')
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_8396/2372296587.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['Medication_Name_Dosage_&_Type'] = df_2['Medication_Name_&_Dosage'].str.cat(


In [82]:
#for i in df['pat_created_date']:
df_2['pat_created_date'] = pd.to_datetime(df_2['pat_created_date']).dt.date

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_8396/2688044427.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_2['pat_created_date'] = pd.to_datetime(df_2['pat_created_date']).dt.date
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_8396/2688044427.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['pat_created_date'] = pd.to_datetime(df_2['pat_created_date']).dt.date


In [83]:
df_2 = df_2.sort_values(by=['pat_created_date','Medication_Name_Dosage_&_Type'])

In [84]:
df_2.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mood_count', 'game_count', 'ah_activity_data',
       'ah_sleep_data', 'phq_count', 'hl_excercise_counts', 'hl_sleep_counts',
       'medication_name', 'dose', 'is_depression_related', 'number_of_refills',
       'Medication_Name_&_Dosage', 'Medication_Name_Dosage_&_Type'],
      dtype='object')

In [85]:
df_2.head(3)

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,mood_count,...,ah_sleep_data,phq_count,hl_excercise_counts,hl_sleep_counts,medication_name,dose,is_depression_related,number_of_refills,Medication_Name_&_Dosage,Medication_Name_Dosage_&_Type
1162,MTIzNDU2Nzg5MGFiY2RlZlVNZrBxMZ2ktAZSJDpfq3Fqjl...,Casey Dennis,PATIENT_GENDER_FEMALE,1987-04-26,38,2024-01-25,The Groover Clinic,Ann Groover,ONBOARDED,307,...,NaN,7.0,303.0,303.0,alprazolam 0.5 mg oral tablet [xanax],1 tbl,False,1,alprazolam 0.5 mg oral tablet [xanax];;; Dose ...,alprazolam 0.5 mg oral tablet [xanax];;; Dose ...
1163,MTIzNDU2Nzg5MGFiY2RlZlVNZrBxMZ2ktAZSJDpfq3Fqjl...,Casey Dennis,PATIENT_GENDER_FEMALE,1987-04-26,38,2024-01-25,The Groover Clinic,Ann Groover,ONBOARDED,307,...,NaN,7.0,303.0,303.0,alprazolam 0.5 mg oral tablet,1.0 tbl,False,1,alprazolam 0.5 mg oral tablet;;; Dose -1.0 tbl,alprazolam 0.5 mg oral tablet;;; Dose -1.0 tbl...
694,MTIzNDU2Nzg5MGFiY2RlZjFDHdkGo80ym5Xvy7V3NfNKwg...,Jennifer Johnson,PATIENT_GENDER_FEMALE,1987-11-26,37,2024-02-20,The Groover Clinic,MeriDee McArthur,SMS_INVITE_DELIVERY_SENT,1,...,NaN,NaN,1.0,1.0,amphetamine aspartate,NaN,False,1,amphetamine aspartate;;; Dose -*,amphetamine aspartate;;; Dose -*;;; Depressive...


In [86]:
df_2['med_flag'] = 1

# Step 3: Pivot for presence
presence_pivot = df_2.pivot_table(index='patient_id',
                                 columns='Medication_Name_Dosage_&_Type',
                                 values='med_flag',
                                 aggfunc='max',
                                 fill_value=0)

# Step 4: Pivot for number of refills
refill_pivot = df_2.pivot_table(index='patient_id',
                               columns='Medication_Name_Dosage_&_Type',
                               values='number_of_refills',
                               aggfunc='max',  # max or sum depending on logic
                               fill_value=0)

# Step 5: Rename columns to differentiate
presence_pivot.columns = [f"{col}__present" for col in presence_pivot.columns]
refill_pivot.columns = [f"{col}__refills" for col in refill_pivot.columns]

# Step 6: Combine both pivot tables
combined = pd.concat([presence_pivot, refill_pivot], axis=1)

# Optional: sort columns so each medication’s present & refill columns appear together
combined = combined.reindex(sorted(combined.columns), axis=1)

# Step 7: Add patient metadata (optional)
patient_info = df_2.drop_duplicates(subset='patient_id')[[
    'patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
    'pat_created_date', 'practice_name', 'provider_name', 'onboarding_status', 
    'mood_count','game_count','ah_activity_data','ah_sleep_data','phq_count', 
    'hl_excercise_counts','hl_sleep_counts'
]]

# Final merge
final_df = pd.merge(patient_info, combined, on='patient_id', how='left')

In [87]:
final_df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mood_count',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills'],
      dtype='object', length=1888)

In [88]:
final_df[final_df['preferred_name'] == 'Casey Dennis']

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,mood_count,...,zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__present,zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills,zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present,zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills,zyprexa;;; Dose -*;;; Depressive- False__present,zyprexa;;; Dose -*;;; Depressive- False__refills,zyprexa;;; Dose -10 mg;;; Depressive- False__present,zyprexa;;; Dose -10 mg;;; Depressive- False__refills,zyprexa;;; Dose -5 mg;;; Depressive- False__present,zyprexa;;; Dose -5 mg;;; Depressive- False__refills
0,MTIzNDU2Nzg5MGFiY2RlZlVNZrBxMZ2ktAZSJDpfq3Fqjl...,Casey Dennis,PATIENT_GENDER_FEMALE,1987-04-26,38,2024-01-25,The Groover Clinic,Ann Groover,ONBOARDED,307,...,0,0,0,0,0,0,0,0,0,0


In [89]:
final_df.shape

(430, 1888)

In [90]:
final_df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mood_count',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills'],
      dtype='object', length=1888)

In [91]:
#final_df.to_csv('Patient_Level_MDD_V1.csv')

In [92]:
df_3 = final_df

In [93]:
list_columns = df_3.columns

In [94]:
print(list_columns)

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mood_count',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills'],
      dtype='object', length=1888)


In [95]:
len(list_columns)

1888

In [96]:
terms = ['dose', 'depressive', 'present']

# Filter strings where ALL terms are present (case-insensitive)
filtered = [text for text in list_columns if all(term.lower() in text.lower() for term in terms)]

print(len(filtered))


936


In [97]:
df_3['total_drugs'] = df_3[filtered].sum(axis=1)

In [98]:
df_3.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mood_count',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills', 'total_drugs'],
      dtype='object', length=1889)

In [99]:
col = 'total_drugs'
cols = list(df_3.columns)
cols.insert(9, cols.pop(cols.index(col)))
df_3 = df_3[cols]

In [100]:
df_3.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'total_drugs',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills'],
      dtype='object', length=1889)

In [101]:
df_3.sort_values(by='total_drugs',inplace=True)

In [102]:
#df_3.to_csv('Patient_Level_MDD_V3.csv')

In [103]:
df_2.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mood_count', 'game_count', 'ah_activity_data',
       'ah_sleep_data', 'phq_count', 'hl_excercise_counts', 'hl_sleep_counts',
       'medication_name', 'dose', 'is_depression_related', 'number_of_refills',
       'Medication_Name_&_Dosage', 'Medication_Name_Dosage_&_Type',
       'med_flag'],
      dtype='object')

### Change Log 1: On June 25, Erwin asked to only pull drugs that are related to Depression. So that ask is done in Path 1 logic, the other
### is in Path 2 logic. If Path 1 (removal of depression) is not required then use Path 2. 

In [104]:
list_features = ['patient_id','preferred_name','gender','date_of_birth','pat_age','pat_created_date','practice_name',
                 'provider_name','onboarding_status','mood_count', 'game_count', 'ah_activity_data','ah_sleep_data', 
                 'phq_count', 'hl_excercise_counts', 'hl_sleep_counts','Medication_Name_Dosage_&_Type','number_of_refills']
# Path 1
df_3 = df_2.loc[df_2['is_depression_related'] == True,list_features]
    # The shape of this path is - (1525, 11)
    # The patients count - 407

#Path 2
#df_3 = df_2[['patient_id','preferred_name','gender','date_of_birth','pat_age',
#            'pat_created_date','practice_name','provider_name','onboarding_status',
#            'Medication_Name_Dosage_&_Type','number_of_refills']]
    # The shape of this path - (3333,11)
    # The patient count - 457

In [105]:
len(df_3['patient_id'].unique())

392

In [106]:
#import pandas as pd  

'''
# Sample data creation for illustration  
data = {  
    'patient_id': [1, 1, 2, 2, 3],  
    'preferred_name': ['John', 'John', 'Alice', 'Alice', 'Bob'],  
    'gender': ['M', 'M', 'F', 'F', 'M'],  
    'date_of_birth': ['1990-01-01', '1990-01-01', '1985-05-15', '1985-05-15', '2000-12-12'],  
    'pat_age': [31, 31, 36, 36, 21],  
    'pat_created_date': ['2020-01-01', '2020-01-01', '2019-06-01', '2019-06-01', '2021-07-07'],  
    'practice_name': ['Practice A', 'Practice A', 'Practice B', 'Practice B', 'Practice C'],  
    'provider_name': ['Dr. Smith', 'Dr. Smith', 'Dr. Jones', 'Dr. Jones', 'Dr. Taylor'],  
    'onboarding_status': ['Completed', 'Completed', 'Pending', 'Pending', 'Completed'],  
    'Medication_Name_Dosage_&_Type': ['Med1', 'Med2', 'Med3', 'Med4', 'Med5'],  
    'number_of_refills': [3, 1, 5, 2, 4]  
}  
  
# Create dataframe  
df = pd.DataFrame(data)
'''
# Step 1: Sort by patient + refills descending
df_3_sorted = df_3.sort_values(by=['patient_id', 'number_of_refills'], ascending=[True, False])

# Step 2: Enumerate drug ranks per patient
df_3_sorted['drug_rank'] = df_3_sorted.groupby('patient_id').cumcount() + 1

# Step 3: Pivot to wide format
pivoted = df_3_sorted.pivot(index='patient_id', 
                          columns='drug_rank', 
                          values=['Medication_Name_Dosage_&_Type', 'number_of_refills'])

# Step 4: Flatten multi-index columns and interleave Drug + Refill
drugs = pivoted['Medication_Name_Dosage_&_Type']
refills = pivoted['number_of_refills']

# Reconstruct interleaved column order
interleaved_cols = []
for i in sorted(drugs.columns):
    interleaved_cols.append(('Medication_Name_Dosage_&_Type', i))
    interleaved_cols.append(('number_of_refills', i))

# Reorder pivoted columns
pivoted = pivoted[interleaved_cols]

# Rename columns to 'Drug {i}' and 'Drug {i} Refills'
pivoted.columns = [f"Drug {i//2 + 1}" if j % 2 == 0 else f"Drug {i//2 + 1} Refills" 
                   for i, j in enumerate(range(len(pivoted.columns)))]

# Add Patient ID back
pivoted.reset_index(inplace=True)

pivoted.reset_index(inplace=True)

# Step 5: Merge with static patient data
patient_info_cols = [
    'patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
    'pat_created_date', 'practice_name', 'provider_name', 'onboarding_status',
    'mood_count', 'game_count', 'ah_activity_data','ah_sleep_data','phq_count', 
    'hl_excercise_counts', 'hl_sleep_counts'
]
patient_info_df = df_3[patient_info_cols].drop_duplicates(subset='patient_id')

final_df = pd.merge(patient_info_df, pivoted, on='patient_id', how='left')

# Count number of non-null drugs per patient
drug_cols = [col for col in final_df.columns if col.startswith("Drug ") and "Refills" not in col]
final_df["total_drugs"] = final_df[drug_cols].notna().sum(axis=1)

# Sort by number of drugs (descending)
final_df = final_df.sort_values(by="total_drugs", ascending=True).reset_index(drop=True)


In [107]:
final_df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'mood_count', 'game_count', 'ah_activity_data',
       'ah_sleep_data', 'phq_count', 'hl_excercise_counts', 'hl_sleep_counts',
       'index', 'Drug 1', 'Drug 1 Refills', 'Drug 2', 'Drug 2 Refills',
       'Drug 3', 'Drug 3 Refills', 'Drug 4', 'Drug 4 Refills', 'Drug 5',
       'Drug 5 Refills', 'Drug 6', 'Drug 6 Refills', 'Drug 7',
       'Drug 7 Refills', 'Drug 8', 'Drug 8 Refills', 'Drug 9',
       'Drug 9 Refills', 'Drug 10', 'Drug 10 Refills', 'Drug 11',
       'Drug 11 Refills', 'Drug 12', 'Drug 12 Refills', 'Drug 13',
       'Drug 13 Refills', 'Drug 14', 'Drug 14 Refills', 'Drug 15',
       'Drug 15 Refills', 'Drug 16', 'Drug 16 Refills', 'Drug 17',
       'Drug 17 Refills', 'Drug 18', 'Drug 18 Refills', 'Drug 19',
       'Drug 19 Refills', 'Drug 20', 'Drug 20 Refills', 'Drug 21',
       'Drug 21 Refills', 'Drug 22'

In [108]:
#df_2[df_2['patient_id'] == 'MTIzNDU2Nzg5MGFiY2RlZnwsg6vkdtdByEewlUR+uFuaW1X0lr8oOR/VDtgj0NydbxsauajItWzN16IOl3B+TuUc76VjtdYXkvbN6azlhAXAsTgRbzJd/89fOkaLI9n1']

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,mood_count,...,phq_count,hl_excercise_counts,hl_sleep_counts,medication_name,dose,is_depression_related,number_of_refills,Medication_Name_&_Dosage,Medication_Name_Dosage_&_Type,med_flag


In [109]:
if 'index' in final_df.columns:
    final_df.drop(columns='index', inplace=True)

# Step 2: Move 'total_drugs' to the position where 'index' was (before 'Drug 1')
# Find current index of 'Drug 1'
drug1_pos = final_df.columns.get_loc('Drug 1') if 'Drug 1' in final_df.columns else len(final_df.columns)

# Reorder columns
cols = final_df.columns.tolist()
cols.insert(drug1_pos, cols.pop(cols.index('total_drugs')))
final_df = final_df[cols]

In [110]:
#final_df.to_csv('patient_level_MDD_V4.csv')

In [111]:
final_df

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,mood_count,...,Drug 21,Drug 21 Refills,Drug 22,Drug 22 Refills,Drug 23,Drug 23 Refills,Drug 24,Drug 24 Refills,Drug 25,Drug 25 Refills
0,MTIzNDU2Nzg5MGFiY2RlZuMRlPmpukzq08G/QH9RMSTHcx...,Chloe Williams,PATIENT_GENDER_FEMALE,1986-01-14,39,2025-02-05,Matthew A Berger MD PC,Abigail Skene,ONBOARDED,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MTIzNDU2Nzg5MGFiY2RlZuKiG15L5sEUS/tqUXL1kEPHE2...,Lisa West,PATIENT_GENDER_MALE,1994-10-23,30,2025-01-06,Visionary Psychiatry,Ana Tattersall Cocke,ONBOARDED,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MTIzNDU2Nzg5MGFiY2RlZhdOHX6+warWC8DrZd2bUbktQV...,Matthew Edwards,PATIENT_GENDER_FEMALE,2006-07-27,18,2025-03-04,Visionary Psychiatry,Catherine Hensy,PASSWORD_CREATED,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MTIzNDU2Nzg5MGFiY2RlZikh3jGz443X0p9QD2rCRWwniy...,Isabella Walters,PATIENT_GENDER_FEMALE,1993-12-22,31,2025-03-04,Visionary Psychiatry,Carmen Kosicek,PASSWORD_CREATED,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MTIzNDU2Nzg5MGFiY2RlZsFg3vxh/rKUHFRf3xOedLOSJY...,Stephanie Smith,PATIENT_GENDER_FEMALE,1991-11-06,33,2025-01-03,Matthew A Berger MD PC,Amanda Craven,ONBOARDED,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
387,MTIzNDU2Nzg5MGFiY2RlZnVm7q3KsXFa0+1L9ldbQVGRym...,Angela Krause,PATIENT_GENDER_FEMALE,1974-02-26,51,2024-11-29,Matthew A Berger MD PC,Daryl Krouse,ONBOARDED,37,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
388,MTIzNDU2Nzg5MGFiY2RlZgbPUqLewgwJaCnqdTUKhaMBRy...,Adrienne Martinez,PATIENT_GENDER_FEMALE,1955-09-07,69,2024-09-20,Matthew A Berger MD PC,Matthew Berger,PASSWORD_CREATED,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
389,MTIzNDU2Nzg5MGFiY2RlZpB1PPq1vmK94b2B86dLuA1XrG...,Sean French,PATIENT_GENDER_FEMALE,2004-04-08,21,2025-02-04,Authentic Psychiatric Services,Jonathan Hoerl,PASSWORD_CREATED,118,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
390,MTIzNDU2Nzg5MGFiY2RlZu50m1vEcVPfCyh2lfYCg8j145...,Joseph Collins,PATIENT_GENDER_FEMALE,1980-03-16,45,2024-10-21,Matthew A Berger MD PC,Melanie Priebe,ONBOARDED,44,...,venlafaxine hcl er 75 mg cap;;; Dose -*;;; Dep...,1,venlafaxine hydrochloride;;; Dose -300 mg;;; D...,1,NaN,NaN,NaN,NaN,NaN,NaN


In [112]:
final_df.to_csv('patient_level_MDD_V7.csv')